# Actin-channel correction

Apply a ratiometric correction to the actin (LifeAct) fluorescence channel to compensate for photobleaching and acquisition drift across the timelapse. Outputs the corrected actin channel as part of the segmentation-ready stack.

In [ ]:


import bioio_ome_tiff
import bioio_tifffile


In [ ]:
input_dirpath = Path(input())

In [ ]:
actinch = 2

proc_dirpath = utils.get_proc_dirpath(input_dirpath)
output_dirpath = proc_dirpath / dn.labbkit_seg_dirname / 'caax_cell_bgsbratiocorractin_seg'
output_dirpath.mkdir(exist_ok=True)

In [ ]:
imgpaths = [path for path in input_dirpath.glob('*.ome.tif')]

for imgpath in tqdm(imgpaths):
    
    if not (output_dirpath / imgpath.name).is_file():
        
        img_file = BioImage(imgpath, open=bioio_ome_tiff.Reader)
        img = img_file.data

        # correct actin channel image
        actin = img[:, actinch, np.newaxis, :, :, :]
        actin = sb.subtract_median(actin)
        actin = tc.timelapse_simpleratio(actin)
        img[:, actinch, np.newaxis, :, :, :] = actin
                
        ome_metadata = utils.construct_ome_metadata(img, img_file)
        OmeTiffWriter.save(img, output_dirpath / imgpath.name, ome_xml=ome_metadata)
    
print('Done!')